# CS 3110/5110: Data Privacy

## Homework 2


In [14]:
# Load the data and libraries
import pandas as pd
import numpy as np

adult = pd.read_csv(
    "https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv"
)
adult = adult.dropna()

## Question 3 (10 points)

Write code to determine the `Education-Num` of the individual named Ardyce Golby by performing a differencing attack. Your code should _only_ use aggregate data to find Ardyce's education number.


In [15]:
def ardyce_education():
    # Differencing Attack
    # To accomplish this we need to get the summation of everyone's
    # education-num values both WITH and WITHOUT including Ardyce.
    # We assume we can only get the MEAN of the data, and therefore do not use
    # .sum()
    education_sum = adult["Education-Num"].mean() * len(adult)
    education_sum_ardyce_removed = adult[adult["Name"] != "Ardyce Golby"][
        "Education-Num"
    ].mean() * (len(adult) - 1)

    education_ardyce = education_sum - education_sum_ardyce_removed
    return education_ardyce

In [16]:
# TEST CASE for Question 3
assert ardyce_education() == 12

## Question 1 (20 points)

Implement a more efficient version of `is_k_anonymous`. The inefficient implementation, taken from the textbook, appears below.

**Hint**: use the [`value_counts`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.value_counts.html) or `group_by` functions, and make sure no count is less than $k$.


In [17]:
# Checking for k-Anonymity, taken from the textbook
# def is_k_anonymous(k, qis, df):
#     for index, row in df.iterrows():
#         query = ' & '.join([f'`{col}` == "{row[col]}"' for col in qis])
#         rows = df.query(query)
#         if (rows.shape[0] < k):
#             return False
#     return True

In [18]:
# Checking for k-anonymity more efficiently
def is_k_anonymous(k, qis, df):
    """Returns true if df satisfies k-Anonymity for the quasi-identifiers
    qis. Returns false otherwise."""

    # Group the provided dataframe by the quasi-identifiers
    df_grouped = df.groupby(qis)

    # Check if all groups have at least k members
    return all(df_grouped.size() >= k)

In [19]:
# TEST CASES for question 1

assert not is_k_anonymous(2, ["Age"], adult)
assert is_k_anonymous(1, ["Age"], adult)
assert is_k_anonymous(1, ["Age", "Occupation"], adult)

## Question 2 (10 points)

Consider the definition of `generalize` below, taken from the textbook. The function takes a dataframe `df` and a dictionary `depths` that describes how much to generalize each column of `df`. Generalizing a column to a depth of $n$ replaces the $n$ least-significant digits of each number in that column by zeroes. For example, we could generalize column `A` by making its least-significant digit a 0 and column `B` by doing the same for 2 digits with the following depth specification:


In [20]:
depths = {"A": 1, "B": 2}
# What this does:
# generalizes column A with 1 digit generalized
# generalizes column B with 2 digits generalized

In [21]:
def generalize(df, depths):
    return df.apply(
        lambda x: x.apply(
            lambda y: int(
                int(y / (10 ** depths[x.name])) * (10 ** depths[x.name])
            )
        )
    )

Using the `generalize` function, generalize the `Age` column of the `adult` dataset to a depth of 1. Drop the other columns of the dataset. Your result should achieve $k$-Anonymity for $k=20$.


In [22]:
# Use the generalize function to accomplish k-anonymity.
# we just need to use the premade function here with different test values.
def generalize_adult_age():
    # Set the "generalization depth" for the Age column to 1.
    depths = {"Age": 1}

    # This applies the generalization and drops the remaining columns that are
    # not generalized as well.
    return generalize(adult[["Age"]], depths)

In [23]:
assert is_k_anonymous(20, ["Age"], generalize_adult_age())

## Question 3 (10 points)

Using the `generalize` function, generalize the `Age` and `Zip` columns of the `adult` dataset in order to achieve $k$-Anonymity for $k=5$. Your result should drop other columns besides these two.


In [24]:
# The values here were chosen via trial and error, testing different values and
# then using the debug prints in the next cell to inform other choices.


def generalize_adult_age_zip():
    depths = {"Age": 2, "Zip": 2}

    return generalize(adult[["Age", "Zip"]], depths)

In [25]:
# Print the smallest and largest groupby sizes on age and zip
grouped = generalize_adult_age_zip().groupby(["Age", "Zip"]).size()
print("Smallest group size:", grouped.min())
print("Largest group size:", grouped.max())

# Print out which groups have the smallest size
print("Group(s) with the smallest size:")
print(grouped[grouped == grouped.min()])

# # print out all the unique age values present in the generalized dataset
# print("Unique age values:", generalize_adult_age_zip()["Age"].unique())
# # print out all the unique zip values present in the generalized dataset
# print("Unique zip values:", generalize_adult_age_zip()["Zip"].unique())

Smallest group size: 14
Largest group size: 51
Group(s) with the smallest size:
Age  Zip  
0    71100    14
dtype: int64


In [27]:
assert is_k_anonymous(5, ["Age", "Zip"], generalize_adult_age_zip())

## Question 4 (30 points)

In 1-4 sentences each, answer the following:

1. How much generalization was required to achieve $k=5$ in question 3?
2. Does this level of generalization significantly impact the utility of the $k$-Anonymized data? Why or why not?
3. Why is generalizing the `adult` dataset so challenging? (**Hint**: consider outliers)
4. Is there another approach, in addition to our simple generalization method, that might work better?
5. What is a simple method for generalizing the `Occupation` column?


1. A large amount of generalization is required: Either {**1 digit for Age** and **5 digits
   for Zip**} or {**2 digits for Age** and **2 digits for Zip**}.

2. Yes this level of generalization significantly impacts the utility of the
   _k_-Anonymized data. In the first case we have to generalize all of the zip codes to a
   single value and in the other we have to reduce all of the age values to a single
   number. In both cases we are essentially completely removing the utility of
   that attribute.

3. Datasets such as the `adult` dataset are challenging to generalize because of
   the presence of a few outliers. Most of the entries in the dataset are able
   to be k-anonymized with smaller depth values; however, the presence of rare
   'Age' and 'Zip' combinations means that even though the majority of the
   entries are easily k-anonymized, the remaining, rarer, entries forced the
   rest of the data to be further generalized in order to achieve full
   k-anonymity.

4. Yes, if we could simply remove certain outlier data points from our dataset,
   then we would more easily achieve k-anonymity without losing too much utility
   in our dataset. Alternatively we could try different generalization
   strategies (such as grouping ages every 20 years instead of 10).

5. One simple method to generalize the `Occupation` column would be to
   categorize everyone as either "employed" or "not currently employed". This
   would easily group people without removing all of the utility of this
   attribute.
